## 第21章　归一化：把数值拉回安全区间

> 本章目标：理解为什么深层网络需要归一化——激活值在层层传递中会发散或消失。掌握 LayerNorm（Transformer 标配）和 RMSNorm（LLaMA/Mistral 的选择）的原理与手写实现。建立"归一化 = 让每层输出的数值始终落在安全区间"的核心直觉。
> 前置知识：第 6 章（矩阵/广播）、第 17 章（均值/方差）、第 20 章（浮点精度）



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math
print("env ready")


### 21.1　为什么需要归一化 —— 深层网络的"数值漂移"

第 20 章你看到 float16 下 `exp(100)` 直接溢出。但在一个 100 层的 Transformer 中，即使每层只把数值放大 1.05 倍——`1.05^100 ≈ 131`——经过 100 层，原本在 [−1, 1] 范围内的激活值会被放大到 ±131。如果每层缩小 0.95 倍——`0.95^100 ≈ 0.006`——信号被压缩到接近 0。

**这就是深层网络的"数值漂移"问题：信号的尺度随层数指数级发散或消失。** 归一化的作用是在每一层之后"强制校准"输出的均值和方差——无论上一层的输出偏到哪里，归一化把它拉回标准位置。

📐 **定义　归一化（Normalization）**：将激活值变换为均值为 0、方差为 1（或类似目标）的分布。核心公式 `(x − μ) / σ`，可选地加回可学习的缩放 γ 和偏移 β。

---


### 21.2　BatchNorm —— 跨样本归一化，Transformer 不用

BatchNorm（BN）在 CNN 时代是标配。它对一个 mini-batch 内的所有样本在**同一特征维度上**求均值和方差。训练时用 batch 统计量，推理时用训练期间累积的全局统计量——这种"训练/推理行为不一致"在序列模型中是个问题。

**为什么 Transformer 不用 BN？** 
- 序列长度可变 → batch 内的句子长度不一，跨序列的统计量不稳定
- 推理时需要全局统计量 → 但 NLP 中测试句子可能与训练句子长度分布完全不同
- 自回归生成时 batch_size=1 → 根本无"batch"可跨

所以 NLP / Transformer 选择了另一种策略——在**单个样本内部**、沿**特征维度**做归一化——这就是 LayerNorm。

💻 **代码　BatchNorm vs LayerNorm 的操作维度对比**


In [ ]:
import numpy as np

# 模拟: (batch=3, features=4) 的激活值
X = np.array([
    [1.0, 2.0, 3.0, 100.0],   # 样本0—最后一维异常大!
    [2.0, 3.0, 4.0, 5.0],
    [3.0, 4.0, 5.0, 6.0],
])

# BatchNorm: 沿 axis=0（跨样本）求每个特征的均值/方差
bn_mean = X.mean(axis=0, keepdims=True)
bn_std = X.std(axis=0, keepdims=True)
X_bn = (X - bn_mean) / (bn_std + 1e-5)

# LayerNorm: 沿 axis=1（每样本内部）求均值/方差
ln_mean = X.mean(axis=1, keepdims=True)
ln_std = X.std(axis=1, keepdims=True)
X_ln = (X - ln_mean) / (ln_std + 1e-5)

print("BatchNorm (跨样本):")
print(X_bn.round(3))
print(f"  每列均值: {X_bn.mean(axis=0).round(3)} (≈0)")
print(f"  每列标准差: {X_bn.std(axis=0).round(3)} (≈1)")

print(f"\nLayerNorm (每样本内部):")
print(X_ln.round(3))
print(f"  每行均值: {X_ln.mean(axis=1).round(3)} (≈0)")
print(f"  每行标准差: {X_ln.std(axis=1).round(3)} (≈1)")

print(f"\n关键差异: BN在样本0的特征3异常值时,会把所有样本的特征3都'拉偏'")
print(f"         LN只在样本0内部归一化,不影响样本1和样本2")


---


### 21.3　LayerNorm —— Transformer 的标配

LayerNorm 对每个 token 的嵌入向量内部做归一化——完全不看同一 batch 内的其他样本。这带来了三个关键优势：序列长度无关（每个 token 独立归一化）、训练推理一致、batch_size=1 也能正常工作。

**前向公式**：
$$y = \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} \cdot \gamma + \beta$$

其中 μ 和 σ² 是对**最后一维（d_model）**的所有值算的。γ 和 β 是可学习参数（shape = d_model），允许模型在归一化后重新调整分布。

**在 Transformer 中的位置**：每个 Sublayer（Attention/FFN）后面——`x = LayerNorm(x + Sublayer(x))`，即 Post-Norm 或 Pre-Norm（现代 Pre-Norm：`x = x + Sublayer(LayerNorm(x))`）。

📐 **定义　LayerNorm**：沿最后一维求 μ 和 σ，对每个 token 独立归一化。`nn.LayerNorm(d_model)`。γ 和 β 可学习，初始化为 1 和 0。

💻 **代码　手写 LayerNorm + 与 PyTorch 对比验证**


In [ ]:
import numpy as np

def layer_norm(x, gamma=None, beta=None, eps=1e-5):
    """
    x: (..., d_model) — 沿最后一维归一化
    """
    mean = x.mean(axis=-1, keepdims=True)
    var = x.var(axis=-1, keepdims=True)
    x_norm = (x - mean) / np.sqrt(var + eps)

    if gamma is not None:
        x_norm = x_norm * gamma
    if beta is not None:
        x_norm = x_norm + beta
    return x_norm

# 测试：Transformer 风格的输入 (batch=2, seq_len=3, d_model=4)
np.random.seed(42)
X = np.random.randn(2, 3, 4)
gamma = np.ones(4)
beta = np.zeros(4)
eps = 1e-5

X_ln = layer_norm(X, gamma, beta, eps)

# 验证：每 token 归一化后均值为 0, 方差≈1
token_means = X_ln.mean(axis=-1)
token_vars = X_ln.var(axis=-1)

print(f"输入 shape: {X.shape}")
print(f"LayerNorm 输出 shape: {X_ln.shape}")
print(f"每个 token 均值 ≈ 0: {np.allclose(token_means, 0.0, atol=1e-6)}")
print(f"每个 token 方差 ≈ 1: {np.allclose(token_vars, 1.0, atol=1e-6)}")

# 与 PyTorch 对比（如果可用）
try:
    import torch
    import torch.nn as nn
    X_t = torch.tensor(X, dtype=torch.float32)
    ln = nn.LayerNorm(4, eps=eps, elementwise_affine=False)
    with torch.no_grad():
        result_torch = ln(X_t).numpy()
    print(f"\n与 PyTorch 一致: {np.allclose(X_ln, result_torch, atol=1e-5)} ✓")
except ImportError:
    print("\n(PyTorch 未安装，手写实现已验证)")


---


### 21.4　RMSNorm —— LLaMA 的选择：去掉均值，只除 RMS

LayerNorm 做了两件事：减均值（中心化）和除标准差（缩放）。但 LLaMA 和 Mistral 等现代 LLM 发现：**减均值这一步可以去掉——只除以 RMS（Root Mean Square）就够了。**

$$RMSNorm(x) = \frac{x}{\sqrt{\frac{1}{n}\sum x_i^2 + \epsilon}} \cdot \gamma$$

好处：(1) 少算一次均值 → 计算快约 15%；(2) 去掉均值中心化后，在某些任务上训练更稳定；(3) 对稀疏激活（ReLU 后很多零）更友好——因为零值不贡献 RMS，但会贡献均值。

📐 **定义　RMSNorm**：只除 RMS，不减均值。`RMS = sqrt(mean(x²))`。γ 可学习。LLaMA/Mistral/Qwen 标配。

💻 **代码　手写 RMSNorm + 与 LayerNorm 速度对比**


In [ ]:
import numpy as np
import time

def rms_norm(x, gamma=None, eps=1e-5):
    """RMSNorm: 只除 RMS，不减均值"""
    rms = np.sqrt(np.mean(x ** 2, axis=-1, keepdims=True))
    x_norm = x / (rms + eps)
    if gamma is not None:
        x_norm = x_norm * gamma
    return x_norm

def layer_norm(x, gamma=None, beta=None, eps=1e-5):
    mean = x.mean(axis=-1, keepdims=True)
    var = x.var(axis=-1, keepdims=True)
    x_norm = (x - mean) / np.sqrt(var + eps)
    if gamma is not None: x_norm = x_norm * gamma
    if beta is not None: x_norm = x_norm + beta
    return x_norm

# 大规模测试：模拟 Transformer 训练中的归一化开销
np.random.seed(42)
X_big = np.random.randn(32, 128, 4096)  # batch=32, seq=128, d=4096 (LLaMA 规模)
gamma = np.ones(4096)
n_trials = 100

# LayerNorm
t0 = time.perf_counter()
for _ in range(n_trials):
    _ = layer_norm(X_big, gamma, np.zeros(4096))
t1 = time.perf_counter()

# RMSNorm
t2 = time.perf_counter()
for _ in range(n_trials):
    _ = rms_norm(X_big, gamma)
t3 = time.perf_counter()

print(f"LayerNorm: {t1 - t0:.4f}s")
print(f"RMSNorm:   {t3 - t2:.4f}s")
print(f"RMSNorm 快 {(t1 - t0) / (t3 - t2) - 1:.0%}")

# 含极端值对比
x_extreme = np.array([[1.0, 2.0, -5.0, 100.0, 0.5]])
print(f"\n含极端值的向量: {x_extreme}")
print(f"LayerNorm: {(x_extreme - x_extreme.mean()) / x_extreme.std():.3f}")
print(f"RMSNorm:   {x_extreme / np.sqrt(np.mean(x_extreme**2)):.3f}")
print("观察: RMSNorm 保留了符号和相对大小关系, 只是缩放了整体幅度")


> **关键洞察**：RMSNorm 没有改变数据的方向——它只做了一个"统一缩放"，让每个 token 的嵌入向量的 RMS 被标准化到 1 附近。减去均值（中心化）会改变数据的"DC 偏移"，这在某些注意力计算中可能反而有害——RMSNorm 保留了原始方向的同时只控制幅度。

🔗 **AI 连接**：第 30 章 Transformer Block 组装中，每个 Attention 和 FFN 前面都有一个 LayerNorm/RMSNorm。第 29 章 QKV 投影后，token 嵌入的数值漂移正需要归一化来校正。

---


### 21.5　从零实现对比：LayerNorm vs RMSNorm 输出分布

💻 **代码　随机输入上对比两种归一化的输出分布**


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
X = np.random.randn(1000, 128)  # 1000 tokens, d_model=128
gamma = np.ones(128)

# 三种处理
X_raw = X.flatten()
X_ln = layer_norm(X, gamma, np.zeros(128)).flatten()
X_rms = rms_norm(X, gamma).flatten()

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, data, title in zip(axes,
    [X_raw, X_ln, X_rms],
    ['原始 (μ≈0, σ≈1 随机)', 'LayerNorm (μ=0, σ=1)', 'RMSNorm (μ不强制为0)']):
    ax.hist(data, bins=50, density=True, color='steelblue', alpha=0.7, edgecolor='white')
    ax.set_title(f'{title}\nμ={data.mean():.3f}, σ={data.std():.3f}')
    ax.axvline(x=0, color='red', linestyle='--', alpha=0.5)
plt.tight_layout(); plt.show()

print("LayerNorm: 严格 μ=0 σ=1")
print("RMSNorm:   σ≈1 但 μ≠0——保留了原始数据的'偏置'方向")
print("现代 LLM(LLaMA/Mistral)选择 RMSNorm——更快且效果不差")


---


**✏️ 习题**

> 在下方新建代码单元格作答。
1. （概念）BatchNorm 和 LayerNorm 的核心区别是什么？为什么 Transformer 不用 BatchNorm？
2. （概念）RMSNorm 比 LayerNorm 少了哪一步？为什么少这一步反而可能更好？
3. （代码）用 NumPy 手写 LayerNorm（不调用框架），输入 `(4, 8)` 的随机矩阵，验证输出每行均值为 0、方差为 1。与 PyTorch 的 `nn.LayerNorm` 对比结果。
4. （代码）构造一个含极端值 `[1, 2, -3, 50, 0.5]` 的向量，分别用 LayerNorm 和 RMSNorm 归一化。对比输出结果，解释差异。
---
> 🔗 **章末钩子**：归一化把数值拉回安全区间。但还有一个操作如果不做归一化就会直接崩溃——`exp(1000)` 在 float32 下等于 Inf。这就是 softmax 的数值问题。
> 预览：下一章——**Softmax 的死亡与复活**，从 NaN 到数值稳定的 log-sum-exp 技巧。
